# RL RESULTS

Plots and summarizes the `train-rl` run folders under `outputs/runs/<exp_name>/<run>/`, one line or row per exp_name.

# INITIAL IMPORTS

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from src.config import Configuration

# PARAMETERS

In [ ]:
RUNS_ROOT = Configuration.OUTPUT_PATH / "runs"
EXP_NAMES = None  # None loads every exp_name; otherwise a list such as ["smoke-rm-symbolic"]

ROLLING_EPISODES = 25  # smoothing window of the training return
GRID_POINTS = 200  # env-step grid the training returns are resampled on

# LOAD RUNS

Only finished runs, the folders with a `summary.json`, are loaded.

In [ ]:
def load_run(directory):
    return {
        "exp_name": directory.parent.name,
        "name": directory.name,
        "config": yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8")),
        "summary": json.loads((directory / "summary.json").read_text(encoding="utf-8")),
        "eval": pd.read_csv(directory / "eval.csv"),
        "episodes": pd.read_csv(directory / "episodes.csv"),
        "train": pd.read_csv(directory / "train.csv"),
    }


runs_by_exp = {}
for directory in sorted(RUNS_ROOT.glob("*/*/")):
    if EXP_NAMES is not None and directory.parent.name not in EXP_NAMES:
        continue
    if not (directory / "summary.json").exists():
        print(f"Skipping unfinished run {directory.relative_to(RUNS_ROOT)}")
        continue
    runs_by_exp.setdefault(directory.parent.name, []).append(load_run(directory))

if not runs_by_exp:
    raise ValueError(f"No finished runs under {RUNS_ROOT} for EXP_NAMES={EXP_NAMES}")
print(f"Loaded {sum(map(len, runs_by_exp.values()))} runs of {len(runs_by_exp)} exp_names")

One row per seed run, with the `summary.json` key names; `final_eval_` prefixes the final evaluation. Runs without a Reward Machine have no RM return, accept rate or accept-exit agreement.

In [ ]:
# The final eval's seed list and episode count are eval settings, not metrics.
run_table = pd.json_normalize(
    [
        {"exp_name": run["exp_name"], "seed": run["config"]["seed"], **run["summary"]}
        for runs in runs_by_exp.values()
        for run in runs
    ],
    sep="_",
).drop(columns=["final_eval_seeds", "final_eval_episodes"])
run_table

# CURVES

Each line is the mean across the seed runs of one exp_name, and its band is ± one std. The std is the population std (`ddof=0`), so with one seed it is 0 and the band is flat.

In [ ]:
# Default colours first, then dashed and dotted repeats, so up to 30 exp_names stay distinct.
LINE_CYCLE = plt.cycler(linestyle=["-", "--", ":"]) * plt.rcParams["axes.prop_cycle"]


def on_grid(grid, steps, values):
    # NaN outside the run's own steps, so each point averages only the runs that reached it.
    return np.interp(grid, steps, values, left=np.nan, right=np.nan)


def eval_curve(runs, metric):
    # Runs that reuse an exp_name may evaluate at different steps, so use every step any run has.
    grid = np.unique(np.concatenate([run["eval"]["global_step"] for run in runs]))
    columns = {
        run["name"]: on_grid(grid, run["eval"]["global_step"], run["eval"][metric]) for run in runs
    }
    return pd.DataFrame(columns, index=grid)


def training_curve(runs):
    # Episodes end at different steps in each run, so resample them on one step grid.
    grid = np.linspace(0, max(run["summary"]["total_env_steps"] for run in runs), GRID_POINTS)
    columns = {}
    for run in runs:
        episodes = run["episodes"]
        if episodes.empty:
            print(
                f"{run['exp_name']}/{run['name']} finished no episode, so it has no training curve"
            )
            continue
        smoothed = episodes["env_return"].rolling(ROLLING_EPISODES, min_periods=1).mean()
        columns[run["name"]] = on_grid(grid, episodes["global_step"], smoothed)
    return pd.DataFrame(columns, index=grid)


def plot_curves(curves, title):
    _, ax = plt.subplots(figsize=(9, 4))
    ax.set_prop_cycle(LINE_CYCLE)
    for exp_name, curve in curves.items():
        mean, std = curve.mean(axis=1), curve.std(axis=1, ddof=0)
        (line,) = ax.plot(curve.index, mean, label=exp_name)
        ax.fill_between(curve.index, mean - std, mean + std, color=line.get_color(), alpha=0.2)
    ax.set(title=title, xlabel="env steps")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.show()

## Evaluation

Evaluations (`eval_epsilon`-greedy), including the final one at the last step.

In [ ]:
EVAL_METRICS = {
    "exit_rate": "Eval exit rate",
    "mean_env_return": "Eval mean env return",
    "mean_length": "Eval mean episode length",
}

for metric, title in EVAL_METRICS.items():
    plot_curves(
        {exp_name: eval_curve(runs, metric) for exp_name, runs in runs_by_exp.items()}, title
    )

## Training return

Env return of each training episode, as a rolling mean over the last `ROLLING_EPISODES` episodes.

In [ ]:
plot_curves(
    {exp_name: training_curve(runs) for exp_name, runs in runs_by_exp.items()},
    f"Training env return (rolling mean over {ROLLING_EPISODES} episodes)",
)

# SUMMARY TABLE

Mean ± std across the seed runs of each exp_name (`ddof=0`, so 0 with one seed). `seeds` counts distinct seeds, and `steps_to_success` reads `reached/runs` plus the mean step over the runs that reached `success_threshold`. `n/a` marks the RM metrics of an exp_name with only sparse runs, and a selection of only sparse runs has no RM columns at all.

In [ ]:
MEAN_STD_COLUMNS = run_table.columns.drop(["exp_name", "seed", "steps_to_success"])


def mean_std(values):
    values = values.dropna()
    if values.empty:
        return "n/a"
    return f"{round(values.mean(), 3):g} ± {round(values.std(ddof=0), 3):g}"


def success_count(steps):
    reached = steps.dropna()
    text = f"{len(reached)}/{len(steps)}"
    return f"{text}, mean {reached.mean():g}" if len(reached) else text


grouped = run_table.groupby("exp_name")
summary_table = grouped[MEAN_STD_COLUMNS].agg(mean_std)
summary_table["steps_to_success"] = grouped["steps_to_success"].agg(success_count)
summary_table["seeds"] = grouped["seed"].nunique()
summary_table